# Assignment 5

In the previous exercises you have gotten detailed instructions about how to solve a predictive task step-by-step. In this week's exercise it will be the opposite: You are given full freedom to do as you choose, given that you are able to solve the task at hand relatively well, and reflect on your choices. The idea is that this should resemble what would happen in a real-life project, be it in research or in the industry: Then you will have few a priori answers to what preprocessing steps are necessary or what model is the best, but you will have to make choices, revise them as you go, and reflect upon the process afterwards.

### 1.0 Preprocess the dataset as you see fit.

In [3]:
import pandas as pd

# Load the dataset
df = pd.read_csv('Credit.csv')

# Inspect the first few rows
print(df.head())

# Check data types and missing values
print(df.info())
print(df.isnull().sum())

# Drop the 'Income' column
df = df.drop('Income', axis=1)

#Encode categorical variables
df = pd.get_dummies(df, columns=['Own', 'Student', 'Married', 'Region'], drop_first=True)

#Feature scaling
from sklearn.preprocessing import StandardScaler

# Identify numeric columns (excluding 'Balance')
numeric_cols = ['Limit', 'Rating', 'Cards', 'Age', 'Education']

scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

#Splitting the data
from sklearn.model_selection import train_test_split

X = df.drop('Balance', axis=1)
y = df['Balance']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


    Income  Limit  Rating  Cards  Age  Education  Own Student Married Region  \
0   14.891   3606     283      2   34         11   No      No     Yes  South   
1  106.025   6645     483      3   82         15  Yes     Yes     Yes   West   
2  104.593   7075     514      4   71         11   No      No      No   West   
3  148.924   9504     681      3   36         11  Yes      No      No   West   
4   55.882   4897     357      2   68         16   No      No     Yes  South   

   Balance  
0      333  
1      903  
2      580  
3      964  
4      331  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Income     400 non-null    float64
 1   Limit      400 non-null    int64  
 2   Rating     400 non-null    int64  
 3   Cards      400 non-null    int64  
 4   Age        400 non-null    int64  
 5   Education  400 non-null    int64  
 6   Own        

### 2. Fit non-linear models

Like GLM (Generalized Linear Model), GAM (Generalized Additive Model), Random Forest (RF) and XGBoost.

In [5]:
#GLM-linear regression

import statsmodels.api as sm

# Add constant for intercept
X_train_const = sm.add_constant(X_train)
X_test_const = sm.add_constant(X_test)

#Converting to float
X_train_const = X_train_const.astype(float)
X_test_const = X_test_const.astype(float)

# Fit GLM (ordinary least squares)
glm_model = sm.OLS(y_train, X_train_const).fit()
y_pred_glm = glm_model.predict(X_test_const)
y_pred_lr = y_pred_glm

print(glm_model.summary())


                            OLS Regression Results                            
Dep. Variable:                Balance   R-squared:                       0.847
Model:                            OLS   Adj. R-squared:                  0.842
Method:                 Least Squares   F-statistic:                     171.5
Date:                Tue, 29 Apr 2025   Prob (F-statistic):          1.02e-119
Time:                        15:56:46   Log-Likelihood:                -2122.9
No. Observations:                 320   AIC:                             4268.
Df Residuals:                     309   BIC:                             4309.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const          469.3348     26.326     17.828   

In [6]:
#Generalized additive model

import pandas as pd
from patsy import dmatrix
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, r2_score

# Define the numeric features you have
numeric_features = ['Limit', 'Rating', 'Cards', 'Age', 'Education']

# Create spline basis for each numeric feature in train and test
spline_bases_train = []
spline_bases_test = []

for feature in numeric_features:
    spline_train = dmatrix(f"bs({feature}, df=5, degree=3, include_intercept=False)",
                           X_train, return_type='dataframe')
    spline_test = dmatrix(f"bs({feature}, df=5, degree=3, include_intercept=False)",
                          X_test, return_type='dataframe')
    spline_bases_train.append(spline_train)
    spline_bases_test.append(spline_test)

# Concatenate all spline bases horizontally
X_splines_train = pd.concat(spline_bases_train, axis=1)
X_splines_test = pd.concat(spline_bases_test, axis=1)

# Fit OLS model with splines
model = sm.OLS(y_train, sm.add_constant(X_splines_train)).fit()

# Predict on test set
y_pred_gam = model.predict(sm.add_constant(X_splines_test))


In [7]:
#Random forest

from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)


In [8]:
#XG Boost

from xgboost import XGBRegressor

xgb = XGBRegressor(random_state=42)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)


In [11]:
##Evaluating model performance

from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

rmse_gam = np.sqrt(mean_squared_error(y_test, y_pred_gam))
r2_gam = r2_score(y_test, y_pred_gam)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)

print(f"{'Model':<20} {'RMSE':<10} {'R²':<10}")
print("-" * 40)
print(f"{'Linear Regression':<20} {rmse_lr:<10.2f} {r2_lr:<10.3f}")
print(f"{'GAM (splines)':<20} {rmse_gam:<10.2f} {r2_gam:<10.3f}")
print(f"{'Random Forest':<20} {rmse_rf:<10.2f} {r2_rf:<10.3f}")
print(f"{'XGBoost':<20} {rmse_xgb:<10.2f} {r2_xgb:<10.3f}")


Model                RMSE       R²        
----------------------------------------
Linear Regression    220.23     0.710     
GAM (splines)        337.49     0.318     
Random Forest        210.65     0.734     
XGBoost              218.38     0.715     


### 4. Reporting model performance

Reflect on the choices you made: Why did you choose the model(s) you did? What about the hyperparameters? How did you tune them?

The GLM was chosen as a baseline, linear regression is simple, interpretable, and helps set a reference for more complex models. It assumes a linear relationship between predictors and the target. GAMs allow for flexible, smooth non-linear relationships between predictors and the response, while still being interpretable. I used splines to capture possible non-linear patterns that linear regression could miss.
The random forest method can model complex, non-linear relationships and interactions without much preprocessing. It’s robust to outliers and can handle a mix of feature types.
XGBoost is a powerful gradient boosting framework known for its predictive accuracy and efficiency. It often outperforms other models on structured data.

A hyperparameter is a setting you choose before training a machine learning model that controls how the learning process works, such as the number of trees in a random forest or the learning rate in XGBoost. In this assignment I used hyperparameters like the number of trees for the random forest and XGBoost, and the degrees of freedom for splines in the GAM-model. Hyperparameter tuning means searching for the best values for these setting. I started with default values, and could move on to something like grid search or random search to optimize the models.

### 5. Reflection

The Random Forest model performed best with the lowest RMSE and highest R², indicating it captured the data patterns well compared to the other models. I expect similar performance on new data if it comes from the same distribution, because the model was evaluated on a separate test set that simulates unseen data. However, without techniques like cross-validation or testing on completely independent datasets, there is some uncertainty about generalization. Using cross-validation, more extensive hyperparameter tuning, or gathering more diverse data could increase confidence in the model’s robustness and predictive power. I'm not doing cross-validation because I forgot and I'm a bit pressed for time. 